# ST-OMR Meter V5-3I — Background TRAIN Acceptance Gate

Read-only TRAIN acceptance. **LAUNCH yalnız bir kez.** Bağlantı koparsa tekrar bağlanıp yalnız STATUS kullan. Historical Validation, First-30, V5 VAL ve FINAL_HOLDOUT kapalıdır.


## 1 — LAUNCH (yalnız bir kez)


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, os, shutil, subprocess, sys, time

RUNNER_HEAD = "d4bfe01b850839328e1f7ce7149d41bce6111e81"
RUNNER_BLOB = "7133edd1967793a9bdc5d53a9bc185b24b758d23"
GATE_IMPLEMENTATION_HEAD = "844c6673f03635177a39b1ab20ab62e9392d922a"
V53G_HEAD = "b36a9d2f5daade2c3568cac8cbc736ca75ca435f"
REPOSITORY = "khfy7wpr5p-maker/st-omr-training"
RUNNER_REL = "tools/meter_v5_3i_background_runner_v1.py"
MYDRIVE = Path("/content/drive/MyDrive")

if not MYDRIVE.is_dir():
    from google.colab import drive
    drive.mount("/content/drive")

DATA_ROOT = MYDRIVE / "TEST" / "METER_V2_1500_PACKAGE_AB_CLEAN"
CHECKPOINT_ROOT = MYDRIVE / "ST-OMR-METER-SPECIALISTS"
M4A_ROOT = CHECKPOINT_ROOT / "m4a-234-digit-specialist-dataset-freeze-v2"
D10_ROOT = MYDRIVE / "ST-OMR-D10" / "stage7d10-authoritative-562c8fcfabf1b41573f1ef591d88ae65335ce16a"
for name, path in {"DATA_ROOT": DATA_ROOT, "CHECKPOINT_ROOT": CHECKPOINT_ROOT, "M4A_ROOT": M4A_ROOT, "D10_ROOT": D10_ROOT}.items():
    if not path.is_dir():
        raise RuntimeError(f"{name} bulunamadi: {path}")
print("DRIVE/PATH CHECK = PASS")

ANN = DATA_ROOT / "annotations"
SOURCE_REPORT = ANN / "v5_3g_authoritative_rescue_training_report.json"
SOURCE_ENVELOPE = ANN / f"v5_3g_execution_envelope_{V53G_HEAD}.json"
RESCUE_DIR = ANN / "v5_3g_authoritative_rescue_artifacts"
for name, path in {"V5-3G REPORT": SOURCE_REPORT, "V5-3H ENVELOPE": SOURCE_ENVELOPE}.items():
    if not path.is_file():
        raise RuntimeError(f"{name} bulunamadi: {path}")
if not RESCUE_DIR.is_dir():
    raise RuntimeError(f"RESCUE DIR bulunamadi: {RESCUE_DIR}")
print("SOURCE EVIDENCE CHECK = PASS")

CONTROL_DIR = ANN / "v5_3i_background_control"
CONTROL_DIR.mkdir(parents=True, exist_ok=True)
LOCK = CONTROL_DIR / f"launch_{GATE_IMPLEMENTATION_HEAD}.json"
HEARTBEAT = CONTROL_DIR / f"heartbeat_{GATE_IMPLEMENTATION_HEAD}.json"
PROGRESS = CONTROL_DIR / f"progress_{GATE_IMPLEMENTATION_HEAD}.json"
LOG = CONTROL_DIR / f"background_{GATE_IMPLEMENTATION_HEAD}.log"
RESULT = ANN / "v5_3i_train_acceptance_gate_v1.json"

if RESULT.exists():
    raise RuntimeError(f"Existing V5-3I gate report blocks launch: {RESULT}")
if LOCK.exists():
    state = json.loads(LOCK.read_text(encoding="utf-8"))
    raise RuntimeError("V5-3I launch lock already exists; second gate process is forbidden. " f"status={state.get('status')} pid={state.get('pid')}")
print("OUTPUT/LOCK GUARD = PASS")

SOURCE_REPO = Path("/content/st-omr-v5-3i-runner-source")
repo_url = f"https://github.com/{REPOSITORY}.git"
if SOURCE_REPO.exists():
    shutil.rmtree(SOURCE_REPO)
subprocess.check_call(["git", "clone", "--no-checkout", repo_url, str(SOURCE_REPO)])
subprocess.check_call(["git", "-C", str(SOURCE_REPO), "fetch", "origin", RUNNER_HEAD, "--depth", "1"])
fetched = subprocess.check_output(["git", "-C", str(SOURCE_REPO), "rev-parse", "FETCH_HEAD"], text=True).strip()
if fetched != RUNNER_HEAD:
    raise RuntimeError(f"runner FETCH_HEAD mismatch: {fetched}")
subprocess.check_call(["git", "-C", str(SOURCE_REPO), "checkout", "--detach", RUNNER_HEAD])
actual_head = subprocess.check_output(["git", "-C", str(SOURCE_REPO), "rev-parse", "HEAD"], text=True).strip()
if actual_head != RUNNER_HEAD:
    raise RuntimeError(f"runner HEAD mismatch: {actual_head}")
if subprocess.check_output(["git", "-C", str(SOURCE_REPO), "status", "--porcelain"], text=True).strip():
    raise RuntimeError("runner source worktree dirty")
runner_path = SOURCE_REPO / RUNNER_REL
if not runner_path.is_file():
    raise RuntimeError(f"runner missing: {runner_path}")
actual_blob = subprocess.check_output(["git", "-C", str(SOURCE_REPO), "hash-object", RUNNER_REL], text=True).strip()
if actual_blob != RUNNER_BLOB:
    raise RuntimeError(f"runner blob mismatch: {actual_blob}")
runner_source = runner_path.read_text(encoding="utf-8")
compile(runner_source, str(runner_path), "exec")
if runner_source.count("gate.run_train_acceptance_gate_v1(") != 1:
    raise RuntimeError("runner gate-call count changed")
for forbidden in ("run_authoritative_rescue_training_v1(", "run_historical_retention_gate(", "torch.optim.", ".backward(", "optimizer.step("):
    if forbidden in runner_source:
        raise RuntimeError(f"runner contains forbidden training/validation token: {forbidden}")
print("EXACT BACKGROUND RUNNER = PASS")
print("RUNNER HEAD =", RUNNER_HEAD)
print("RUNNER BLOB =", RUNNER_BLOB)

initial = {"schema": "st-omr-meter-v5-3i-background-launch-v1", "gate_implementation_head": GATE_IMPLEMENTATION_HEAD, "runner_head": RUNNER_HEAD, "runner_blob": RUNNER_BLOB, "status": "ALLOCATED", "allocated_at_utc": datetime.now(timezone.utc).isoformat(), "decision": None, "log_path": str(LOG), "heartbeat_path": str(HEARTBEAT), "progress_path": str(PROGRESS), "result_path": str(RESULT)}
payload = (json.dumps(initial, indent=2, sort_keys=True) + "\n").encode("utf-8")
fd = os.open(str(LOCK), os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600)
try:
    os.write(fd, payload)
finally:
    os.close(fd)

log_handle = LOG.open("ab", buffering=0)
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
try:
    proc = subprocess.Popen([sys.executable, "-u", str(runner_path)], stdin=subprocess.DEVNULL, stdout=log_handle, stderr=subprocess.STDOUT, start_new_session=True, close_fds=True, env=env)
finally:
    log_handle.close()

time.sleep(2)
if proc.poll() is not None:
    tail = LOG.read_text(encoding="utf-8", errors="replace").splitlines()[-100:]
    raise RuntimeError("V5-3I background runner exited immediately:\n" + "\n".join(tail))
state = json.loads(LOCK.read_text(encoding="utf-8"))
print("V5-3I BACKGROUND LAUNCH = PASS")
print("PID =", state.get("pid", proc.pid))
print("STATUS =", state.get("status"))
print("LOG =", LOG)
print("HEARTBEAT =", HEARTBEAT)
print("PROGRESS =", PROGRESS)
print("RESULT =", RESULT)
print("Bağlantı koparsa LAUNCH hücresini tekrar çalıştırma; yalnız STATUS hücresini kullan.")


## 2 — STATUS (salt okunur; istediğin kadar)


In [ ]:
from pathlib import Path
import json

GATE_IMPLEMENTATION_HEAD = "844c6673f03635177a39b1ab20ab62e9392d922a"
ANN = Path("/content/drive/MyDrive/TEST/METER_V2_1500_PACKAGE_AB_CLEAN/annotations")
CONTROL_DIR = ANN / "v5_3i_background_control"
LOCK = CONTROL_DIR / f"launch_{GATE_IMPLEMENTATION_HEAD}.json"
HEARTBEAT = CONTROL_DIR / f"heartbeat_{GATE_IMPLEMENTATION_HEAD}.json"
PROGRESS = CONTROL_DIR / f"progress_{GATE_IMPLEMENTATION_HEAD}.json"
LOG = CONTROL_DIR / f"background_{GATE_IMPLEMENTATION_HEAD}.log"
RESULT = ANN / "v5_3i_train_acceptance_gate_v1.json"

if not LOCK.is_file():
    raise RuntimeError("V5-3I launch receipt bulunamadi.")
state = json.loads(LOCK.read_text(encoding="utf-8"))
print("STATUS =", state.get("status"))
print("PID =", state.get("pid"))
print("DECISION =", state.get("decision"))
print("ALLOCATED =", state.get("allocated_at_utc"))
print("STARTED =", state.get("started_at_utc"))
print("COMPLETED =", state.get("completed_at_utc"))
print("ERROR =", state.get("error_type"), state.get("error_message"))
print("RESULT EXISTS =", RESULT.is_file())
if HEARTBEAT.is_file():
    print("HEARTBEAT =", json.loads(HEARTBEAT.read_text(encoding="utf-8")))
if PROGRESS.is_file():
    print("PROGRESS =", json.loads(PROGRESS.read_text(encoding="utf-8")))
if LOG.is_file():
    lines = LOG.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\n--- LOG TAIL (last 120 lines) ---")
    print("\n".join(lines[-120:]))


## 3 — FINAL RECEIPT (salt okunur; sonuç hazır olunca)


In [ ]:
from pathlib import Path
import hashlib, json

GATE_IMPLEMENTATION_HEAD = "844c6673f03635177a39b1ab20ab62e9392d922a"
ANN = Path("/content/drive/MyDrive/TEST/METER_V2_1500_PACKAGE_AB_CLEAN/annotations")
RESULT = ANN / "v5_3i_train_acceptance_gate_v1.json"
LOCK = ANN / "v5_3i_background_control" / f"launch_{GATE_IMPLEMENTATION_HEAD}.json"

if not RESULT.is_file():
    print("FINAL RECEIPT = NOT READY")
else:
    result = json.loads(RESULT.read_text(encoding="utf-8"))
    digest = hashlib.sha256(RESULT.read_bytes()).hexdigest()
    state = json.loads(LOCK.read_text(encoding="utf-8")) if LOCK.is_file() else {}
    print("FINAL RECEIPT = READY")
    print("BACKGROUND STATUS =", state.get("status"))
    print("DECISION =", result.get("decision"))
    print("REASONS =", result.get("acceptance_reasons"))
    for digit in ("2", "3"):
        item = result["per_specialist"][digit]
        v5 = item["v5_train"]
        hist = item["historical_train"]
        metrics = v5["combined_metrics"]
        print(f"{digit}-AI: V5_F1={metrics['f1']} FP={metrics['fp']} FN={metrics['fn']} V5_FROZEN_CORRECT_REGRESSIONS={v5['frozen_correct_regression_count']} HIST_TRAIN_FROZEN_CORRECT_REGRESSIONS={hist['frozen_correct_regression_count']}")
    print("FROZEN STATE BIT IDENTICAL =", result.get("frozen_state_bit_identical"))
    print("ONLY RESCUE PARAMETERS CHANGED =", result.get("only_rescue_parameters_changed"))
    print("HISTORICAL VALIDATION EXECUTED =", result.get("historical_validation_retention_executed"))
    print("FIRST-30 OPENED =", result.get("first30_opened"))
    print("V5 RESERVE OPENED =", result.get("v5_reserve_opened"))
    print("V5 VALIDATION OPENED =", result.get("v5_validation_opened"))
    print("FINAL_HOLDOUT LOCKED =", result.get("final_holdout_locked"))
    print("RETRAINING AUTHORIZED =", result.get("retraining_authorized"))
    print("REPORT SHA256 =", digest)
